In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs
from MolEval import MolEmb 

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a depende

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from qsprpred.data.descriptors.sets import RDKitDescs

def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    #dataset.prepareDataset(
    #feature_calculators=[MorganFP(radius=2, nBits=1024)],
    #recalculate_features=True,
    #shuffle=False
    #)
    #from qsprpred.data.descriptors.sets import RDKitDescs
    
    #rdkit_descs = RDKitDescs()
    
    #dataset.addDescriptors([rdkit_descs])
    
    #dataset.descriptorSets
    return dataset

class Dataset_creator():
    def __init__(self, model_names=['RoBERTa_ZINC'], corr_thrsh= 0.95):
        self.variance = VarianceThreshold(threshold=0.0)
        self.model_names = model_names
        self.corr_thrsh = corr_thrsh
        self.selected_indices = None
        
    def fit_transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)
        
        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)
        
        dataset_no_var_np = self.variance.fit_transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        
        dataset_without_high_corr = self.high_correlation(dataset_without_no_var)
        display(dataset_without_high_corr.shape)
        
        dataset.X = dataset_without_high_corr
        return dataset
        
    def transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)

        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)

        dataset_no_var_np = self.variance.transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        dataset_without_high_corr = dataset_without_no_var[self.selected_indices]
        display(dataset_without_high_corr.shape)

        dataset.X = dataset_without_high_corr
        return dataset

    def high_correlation(self, df: pd.DataFrame):
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        to_drop = [column for column in upper.columns if any(upper[column] > self.corr_thrsh)]
        self.selected_indices = df.columns.difference(to_drop)
        
        return df[self.selected_indices]

        
    def create_embs(self, dataset):
        dataset.df["SMILES"] = dataset.df["Drug"]
        final_emb = pd.DataFrame()
        for model_name in self.model_names:
            extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=dataset.df)
            new_emb, dataset.df = extractor.get_embeddings()
            display(type(new_emb))
            if final_emb.empty:
                final_emb = new_emb
            else:
                final_emb = pd.concat([final_emb, new_emb], axis=1)
        final_emb.columns = final_emb.columns.astype(str)
        display(final_emb)
        return final_emb
    


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("HIV/data/hiv_train_1")

X2_all = load_datasets("HIV/data/hiv_val_1")

X3_all = load_datasets("HIV/data/hiv_test_1")

In [4]:
cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])

In [ ]:
X1_all = cls.fit_transform(X1_all)
X2_all = cls.transform(X2_all)
X3_all = cls.transform(X3_all)


In [ ]:
for i in range(2, 11):
    X1_all = load_datasets(f"HIV/data/hiv_train_{i}")

    X2_all = load_datasets(f"HIV/data/hiv_val_{i}")

    X3_all = load_datasets(f"HIV/data/hiv_test_{i}")
    cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])
    X1_all = cls.fit_transform(X1_all)
    X2_all = cls.transform(X2_all)
    X3_all = cls.transform(X3_all)

    X1_all.X.to_csv(f"HIV/mod_data/X1.{i}")
    X2_all.X.to_csv(f"HIV/mod_data/X2.{i}")
    X3_all.X.to_csv(f"HIV/mod_data/X3.{i}")
    X1_all.y.to_csv(f"HIV/mod_data/y1.{i}")
    X2_all.y.to_csv(f"HIV/mod_data/y2.{i}")
    X3_all.y.to_csv(f"HIV/mod_data/y3.{i}")

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


(22200, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

...

(9806, 4722)

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


(22060, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
X1_all.X.to_csv("HIV/mod_data/X1.1")
X2_all.X.to_csv("HIV/mod_data/X2.1")
X3_all.X.to_csv("HIV/mod_data/X3.1")
X1_all.y.to_csv("HIV/mod_data/y1.1")
X2_all.y.to_csv("HIV/mod_data/y2.1")
X3_all.y.to_csv("HIV/mod_data/y3.1")

In [109]:
import pandas as pd
import numpy as np

# Předpokládám, že load_datasets vrací objekt s atributy getDF(), X a y
# a že getDF() vrací pandas DataFrame.

# Načtení dat pro první iteraci (mimo cyklus)
X1_all = load_datasets("HIV/data/hiv_train_1")
X2_all = load_datasets("HIV/data/hiv_val_1")
X3_all = load_datasets("HIV/data/hiv_test_1")

smiles1 = pd.concat([X1_all.getDF()["Drug"], X2_all.getDF()["Drug"], X3_all.getDF()["Drug"]]).reset_index(drop=True)
X_all = pd.concat([X1_all.X, X2_all.X, X3_all.X])
y_all = pd.concat([X1_all.y, X2_all.y, X3_all.y])

for i in range(2, 11):
    X1_all2 = load_datasets(f"HIV/data/hiv_train_{i}")
    X2_all2 = load_datasets(f"HIV/data/hiv_val_{i}")
    X3_all2 = load_datasets(f"HIV/data/hiv_test_{i}")

    smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index(drop=True)
    display(smiles2)
    
    # Změna: Ignorujeme léky, které nejsou v smiles1 a mažeme je z datasetů
    valid_smiles = smiles2[smiles2.isin(smiles1)] # Filtrujeme smiles2 na platné
    indices_map = [list(smiles1).index(element) for element in valid_smiles] # Používáme filtrované smiles2
    
    # Upravíme původní DataFrames X1_all2, X2_all2, X3_all2
    X1_all2.df[X1_all2.getDF()["Drug"].isin(smiles1)]
    X2_all2.df[X2_all2.getDF()["Drug"].isin(smiles1)]
    X3_all2.df[X3_all2.getDF()["Drug"].isin(smiles1)]

    # Opatrně s indexováním y_all, pokud indices_map může být kratší!
    #if indices_map:
     ###   y_all_selected = y_all.iloc[indices_map]
    #else:
     #   y_all_selected = pd.Series([])
    indices_map = [list(smiles1).index(element) for element in pd.concat([X1_all2.df["Drug"],X2_all2.df["Drug"],X3_all2.df["Drug"]])]
    y_concat = pd.concat([X1_all2.df.Y, X2_all2.df.Y, X3_all2.df.Y])
    y_all_selected = y_all.iloc[indices_map]
    if np.array_equal(y_all_selected.values, y_concat.values):
        print("good")
    else:
        print("Critical error")
        break

    # Opatrně s indexováním X_all, pokud indices_map může být kratší!
    X_res = X_all.iloc[indices_map]

    
    X1_res = X_res.iloc[:X1_all2.getDF().shape[0]] 
    X2_res = X_res.iloc[X1_all2.getDF().shape[0]:-X3_all2.getDF().shape[0]] 
    X3_res = X_res.iloc[-X3_all2.getDF().shape[0]:]

    X1_res.to_csv(f"HIV/mod_data/X1.{i}")
    X2_res.to_csv(f"HIV/mod_data/X2.{i}")
    X3_res.to_csv(f"HIV/mod_data/X3.{i}")
    X1_all2.y.to_csv(f"HIV/mod_data/y1.{i}")
    X2_all2.y.to_csv(f"HIV/mod_data/y2.{i}")
    X3_all2.y.to_csv(f"HIV/mod_data/y3.{i}")


0        CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...
1        C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...
2          Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1
3                                   O=S(=O)(O)CCS(=O)(=O)O
4                               CCOP(=O)(Nc1cccc(Cl)c1)OCC
                               ...                        
40665    COc1ccc(C=C2CN(C)CC3C2=NN(c2ccccc2)C3c2ccc(OC)...
40666        CN1CC(=Cc2cccs2)C2=NN(c3ccccc3)C(c3cccs3)C2C1
40667             CN1CC(=Cc2cccs2)C2=NC(=S)NC(c3cccs3)C2C1
40668      CN1CC(=Cc2cccs2)C2=C(C1)C(c1cccs1)C(C#N)=C(N)O2
40669               CN1CC(=Cc2cccs2)C2=NN(C)C(c3cccs3)C2C1
Name: Drug, Length: 40670, dtype: object

Critical error


In [111]:
    X1_all2 = load_datasets(f"HIV/data/hiv_train_{2}")
    X2_all2 = load_datasets(f"HIV/data/hiv_val_{2}")
    X3_all2 = load_datasets(f"HIV/data/hiv_test_{2}")

In [112]:
    smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index(drop=True)
    display(smiles2.shape)

(40670,)

In [ ]:
    # Změna: Ignorujeme léky, které nejsou v smiles1 a mažeme je z datasetů
    valid_smiles = smiles2[smiles2.isin(smiles1)] # Filtrujeme smiles2 na platné
    indices_map = [list(smiles1).index(element) for element in valid_smiles] # Používáme filtrované smiles2

In [118]:
    #display(valid_smiles.shape)
    display(len(indices_map))

40670

In [120]:
smiles2.isin(smiles1).sum()

40670

In [ ]:
    # Upravíme původní DataFrames X1_all2, X2_all2, X3_all2
    X1_all2.df[X1_all2.getDF()["Drug"].isin(smiles1)]
    X2_all2.df[X2_all2.getDF()["Drug"].isin(smiles1)]
    X3_all2.df[X3_all2.getDF()["Drug"].isin(smiles1)]

In [ ]:
    indices_map = [list(smiles1).index(element) for element in pd.concat([X1_all2.df["Drug"],X2_all2.df["Drug"],X3_all2.df["Drug"]])]
    y_concat = pd.concat([X1_all2.df.Y, X2_all2.df.Y, X3_all2.df.Y])
    y_all_selected = y_all.iloc[indices_map]
    if np.array_equal(y_all_selected.values, y_concat.values):
        print("good")
    else:
        print("Critical error")
        break

    # Opatrně s indexováním X_all, pokud indices_map může být kratší!
    X_res = X_all.iloc[indices_map]

    
    X1_res = X_res.iloc[:X1_all2.getDF().shape[0]] 
    X2_res = X_res.iloc[X1_all2.getDF().shape[0]:-X3_all2.getDF().shape[0]] 
    X3_res = X_res.iloc[-X3_all2.getDF().shape[0]:]

    X1_res.to_csv(f"HIV/mod_data/X1.{i}")
    X2_res.to_csv(f"HIV/mod_data/X2.{i}")
    X3_res.to_csv(f"HIV/mod_data/X3.{i}")
    X1_all2.y.to_csv(f"HIV/mod_data/y1.{i}")
    X2_all2.y.to_csv(f"HIV/mod_data/y2.{i}")
    X3_all2.y.to_csv(f"HIV/mod_data/y3.{i}")

KeyboardInterrupt: 

In [51]:
smiles1["Drug"].nunique()

40564

In [50]:
df["Drug"].nunique()

40565

In [54]:
len(set_smiles)


40564

In [99]:
ind = 0
X3_all = load_datasets("HIV/data/hiv_test_1")
set_smiles = set(smiles1["Drug"])
for index, element in df_add.getDF().iterrows():
    if element["Drug"] not in set_smiles:
        print(ind, element["Drug"])
        X3_all.df.loc[ind] = element
        ind += 1


0 CS(C)=O
1 O=S(c1ccccc1O)S(=O)c1ccccc1O
2 O=S1c2ccccc2Sc2ccccc21
3 O=S1c2ccccc2S(=O)c2ccccc21
4 O=S(c1ccc(Cl)cc1)S(=O)c1ccc(Cl)cc1
5 Cc1ccc2c(c1)S(=O)c1cc(C)ccc1S2
6 CC1S(=O)C(C)S(=O)C(C)S1=O
7 O=S1CCCCS1
8 CSCS(=O)CC(CO)NC(=O)C=Cc1c(C)nc(O)nc1O
9 Cc1c2ccccc2n[c-](CS(=O)c2ccccc2)[n+]1=O
10 Cc1c2ccccc2n[c-](CS(C)=O)[n+]1=O
11 CN(C)C1C(O)=C(C(=O)NCNC(CCS(C)=O)C(=O)O)C(=O)C2(O)C(O)=C3C(=O)c4c(O)cccc4C(C)(O)C3CC12
12 Cc1c(S(=O)c2cc(C(C)(C)C)c(O)c(CN3CCCC3)c2C)cc(C(C)(C)C)c(O)c1CN1CCCC1
13 CC(C)(C)SS(=O)C(C)(C)C
14 O=C1CC2CCCC(C1)S2=O
15 O=S(CC(O)c1ccccc1)c1ccccc1
16 O=S1CC(c2ccccc2)=C(c2ccccc2)CS1
17 CN1CCCCC1CCN1c2ccccc2Sc2ccc(S(C)=O)cc21
18 Cc1ccc(CS(=O)C(Cl)(Cl)c2ccc(C)cc2)cc1
19 O=S(CC=CCS(=O)c1ccccc1)c1ccccc1
20 Cc1nc(O)nc(O)c1C=CC(=O)NC(CO)CS(=O)Cc1ccccc1
21 Cn1c(=O)c(S(C)=O)c(O)c2ccccc21
22 CC(=O)OC1CSS(=O)CC1OC(C)=O
23 O=S(Cc1ccc2c(c1)OCO2)c1ccccc1
24 O=C1OC2CS(=O)CC2O1
25 Cn1c2c(c(=O)n(C)c1=O)CS(=O)c1ccccc1N2
26 CS(=O)(=O)CS(=O)CS(=O)CS(=O)CC(NC(=O)CCC(N)C(=O)O)C(=O)O
27 O=C(c1cc

In [97]:
X3_all = cls.transform(X3_all)

KeyError: "None of [Index(['A2ARDataset_00000', 'A2ARDataset_00001', 'A2ARDataset_00002',\n       'A2ARDataset_00003', 'A2ARDataset_00004', 'A2ARDataset_00005',\n       'A2ARDataset_00006', 'A2ARDataset_00007', 'A2ARDataset_00008',\n       'A2ARDataset_00009',\n       ...\n       'A2ARDataset_39858', 'A2ARDataset_40026', 'A2ARDataset_40029',\n       'A2ARDataset_40031', 'A2ARDataset_40033', 'A2ARDataset_40038',\n       'A2ARDataset_40041', 'A2ARDataset_40273', 'A2ARDataset_40410',\n       'A2ARDataset_40626'],\n      dtype='object', length=10361)] are in the [index]"

In [98]:
X3_all.df

,QSPRID,Y,Drug,Y_original
0,A2ARDataset_00000,False,O=[N+]([O-])c1ccccc1SSc1ccccc1[N+](=O)[O-],0
1,A2ARDataset_00001,False,CC(C)(CCC(=O)O)CCC(=O)O,0
2,A2ARDataset_00002,False,CN(C)C1=[S+][Zn-2]2(S1)SC(N(C)C)=[S+]2,0
3,A2ARDataset_00003,False,CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)NC(CCC(=O)...,0
4,A2ARDataset_00004,False,[N-]=[N+]=CC(=O)OCC(N)C(=O)O,0
...,...,...,...,...
10356,A2ARDataset_40038,False,COc1ccc2nc3ccc(OC)cc3c([S+]([O-])Cc3ccccc3)c2c1,0
10357,A2ARDataset_40041,False,COc1ccc2nc3ccc(OC)cc3c([S+]([O-])Cc3ccc([N+](=...,0
10358,A2ARDataset_40273,False,O=C1Sc2ccccc2C(=O)N2C[S+]([O-])CC12,0
10359,A2ARDataset_40410,False,Cc1cn(COCC[S+]([O-])c2ccccc2)c(=O)[nH]c1=O,0
